# Populate the Exact-Greedy Sub-300 Archive in Parallel

Run independent exact-greedy/tabu searches in worker processes. Seed construction and all SQLite reads/writes remain in the parent process. The archive-backed construction is consumed through the standard `construct(graph)` interface; a full 250-seed pass forms a natural barrier before descendants can enter the next pass.

In [1]:
# Imports and Project Root

from pathlib import Path

import numpy as np

from ramsey import (
    REnvironmentConfig,
    RGraph,
    RProblem,
    RSQLiteArchive,
    RTabuMemoryConfig,
)
from ramsey.RArchiveBatchParallel import (
    RArchiveBatchParallel,
    RArchiveBatchParallelConfig,
)
from ramsey.RConstructionArchiveQueue import (
    RArchiveQueueConstruction,
)
from ramsey.RSearchParallel import (
    RExactGreedyProcessConfig,
    RExactGreedyProcessPool,
)

# resolve the project root Path
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root "
        "or notebooks directory."
    )

In [2]:
# Experiment Configuration

RANDOM_SEED = 202_608_062
N_VERTICES = 43

RUN_NAME = "greedy-exact-sub-300-parallel-002"
TARGET_SUB_300_COLORINGS = 1_500  # produce colorings until the database has this amount
MAXIMUM_ATTEMPTS = 2_000          # 
START_ITERATION = 0               # start rollout enumeration from this value

TARGET_MAXIMUM_SCORE = 299
ARCHIVE_SEED_SCORE_LIMIT = 399
MINIMUM_ARCHIVE_SEEDS = 100
ACTIVE_SEED_POOL_SIZE = 250

SEARCH_STEPS = 500
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

MAX_WORKERS = 4
ACTION_SEED_BASE = RANDOM_SEED + 1_000_000

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [3]:
# Graph, Archive, and Live Seed Queue

rng = np.random.default_rng(RANDOM_SEED)
problem = RProblem.r55(n_vertices=N_VERTICES)
graph = RGraph(problem)

existing_archive = globals().get("archive")

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(DATABASE_PATH)

eligible_seed_count = archive.coloring_count_in_score_range(
    maximum_score=ARCHIVE_SEED_SCORE_LIMIT,
    graph=graph,
)
existing_target_count = archive.coloring_count_in_score_range(
    maximum_score=TARGET_MAXIMUM_SCORE,
    graph=graph,
)

if eligible_seed_count < MINIMUM_ARCHIVE_SEEDS:
    archive.close()
    raise RuntimeError(
        f"Expected at least {MINIMUM_ARCHIVE_SEEDS} archived "
        f"sub-{ARCHIVE_SEED_SCORE_LIMIT + 1} seeds; "
        f"found {eligible_seed_count}."
    )

construction = RArchiveQueueConstruction(
    archive=archive,
    rng=rng,
    maximum_score=ARCHIVE_SEED_SCORE_LIMIT,
    limit=ACTIVE_SEED_POOL_SIZE,
)

print("Database:", DATABASE_PATH.resolve())
print("Archive best:", archive.best_score(graph))
print("Eligible archived seeds:", eligible_seed_count)
print("Existing sub-300 colorings:", existing_target_count)
print("Target sub-300 colorings:", TARGET_SUB_300_COLORINGS)
print("Active seed pool:", ACTIVE_SEED_POOL_SIZE)
print("Workers:", MAX_WORKERS)

Database: C:\code\RamseyNumber\data\ramsey_colorings.sqlite3
Archive best: 129
Eligible archived seeds: 2173
Existing sub-300 colorings: 1322
Target sub-300 colorings: 1500
Active seed pool: 250
Workers: 4


In [4]:
# Parallel Exact-Greedy Search and Archive Batch

process_config = RExactGreedyProcessConfig(
    problem=problem,
    environment=REnvironmentConfig(
        max_steps=SEARCH_STEPS,
        use_aspiration=True,
    ),
    memory=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)

search_pool = RExactGreedyProcessPool(
    process_config,
    max_workers=MAX_WORKERS,
)

batch = RArchiveBatchParallel(
    graph=graph,
    construction=construction,
    archive=archive,
    search_pool=search_pool,
)

batch_config = RArchiveBatchParallelConfig(
    run_name=RUN_NAME,
    target_count=TARGET_SUB_300_COLORINGS,
    maximum_attempts=MAXIMUM_ATTEMPTS,
    batch_size=ACTIVE_SEED_POOL_SIZE,
    action_seed_base=ACTION_SEED_BASE,
    maximum_score=TARGET_MAXIMUM_SCORE,
    start_iteration=START_ITERATION,
    save_out_of_range=False,
)

In [5]:
# Pass-Level Reporting

progress = {
    "best": archive.best_score(graph),
}

def report_parallel_pass(parallel_pass):
    initial_scores = parallel_pass.initial_scores
    best_scores = parallel_pass.best_scores
    reductions = initial_scores - best_scores

    for attempt in parallel_pass.attempts:
        score = attempt.search_result.best_score

        if progress["best"] is None or score < progress["best"]:
            progress["best"] = score
            print(
                f"NEW RECORD: {score} | "
                f"attempt={attempt.attempt} | "
                f"archive_id={attempt.archive_record.coloring_id if attempt.archive_record else '-'}"
            )

    print(
        f"Pass {parallel_pass.pass_number:2d} | "
        f"attempts={parallel_pass.attempt_count:3d} | "
        f"seed_mean={initial_scores.mean():6.1f} | "
        f"best_mean={best_scores.mean():6.1f} | "
        f"best_median={np.median(best_scores):5.1f} | "
        f"min={best_scores.min():3d} | "
        f"reduction={reductions.mean():6.1f} | "
        f"improved={parallel_pass.improved_count:3d}/{parallel_pass.attempt_count:3d} | "
        f"sub300={parallel_pass.in_score_range_count:3d} | "
        f"new={parallel_pass.new_unique_count:3d} | "
        f"archive={parallel_pass.eligible_count:4d}/{TARGET_SUB_300_COLORINGS:4d} | "
        f"time={parallel_pass.elapsed_seconds:6.1f}s | "
        f"rate={parallel_pass.attempts_per_second:5.2f}/s"
    )

In [ ]:
# Populate to 1,500 Unique Sub-300 Colorings

try:
    batch_result = batch.populate(
        batch_config,
        observer=report_parallel_pass,
    )
finally:
    search_pool.close()

print()
print("Attempts completed:", batch_result.attempts_completed)
print("Parallel passes:", len(batch_result.passes))
print("Target reached:", batch_result.target_reached)
print("Initial eligible:", batch_result.initial_eligible_count)
print("Final eligible:", batch_result.final_eligible_count)
print("New eligible:", batch_result.new_eligible_colorings)
print("Best attempt score:", batch_result.best_score)
print("Archive best:", archive.best_score(graph))
print("Elapsed:", f"{batch_result.elapsed_seconds:.3f} seconds")

if batch_result.attempts_completed:
    print(
        "Mean time per attempt:",
        f"{batch_result.elapsed_seconds / batch_result.attempts_completed:.3f} seconds",
    )
    print(
        "Effective throughput:",
        f"{batch_result.attempts_completed / batch_result.elapsed_seconds:.2f} attempts/second",
    )